# 09. Run the whole pipeline on an UNSEEN video (scene2)

This notebook does **no training**. It takes the videos in
`../data/validation/scene2` and pushes them through every stage of the
pipeline using the parts that were built on **scene1 (video1)**:

| stage | reused scene1 artifact |
|---|---|
| mouse keypoints | the SuperAnimal weights **video-adapted on video1's pseudolabels** (`pseudo_*/checkpoints/snapshot-hrnet_w32-004.pt` + the matching detector), run with `video_adapt=False` so scene2 is *not* re-adapted |
| calibration | `lockbox_calibration/scene1/lockbox_calibration.toml` (same `CameraGroup`) |
| lockbox keypoints | the three per-view DLC projects under `lockbox_dlc/scene1/perview` |
| 3D inpainting (optional) | `nn_output/scene1/pose_inpainter.pt` |
| state schema | `lockbox_dlc/scene1/state_schema.json` |

At the end we score the predicted lockbox **stage** against scene2's own
ground-truth `…_labels.csv`, a genuine generalization test on a video the models
never saw.

> **Two assumptions worth checking.**
> 1. **Calibration reuse is only valid if the camera rig did not move between the
> two sessions.** scene1 = 2021-06-18, scene2 = 2021-06-17, same rig/mouse, so
> this is the intended use, but if the 3D looks shifted, that points straight
> at extrinsics, and you'd re-click 6 points (Notebook 01) for scene2.
> 2. The mechanism to ground-truth-column mapping in the eval section
> (`lever1 to lever`, `slider1 to stick`, `ball1 to ball`, `cover1 to sliding_door`) is set
> by name. Verify it matches your lockbox.

## 1. Imports & paths

In [1]:
import os, gc, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import deeplabcut
from deeplabcut.modelzoo.video_inference import video_inference_superanimal
from aniposelib.cameras import CameraGroup, Camera

warnings.filterwarnings('ignore')
print('DLC', deeplabcut.__version__, '| OpenCV', cv2.__version__)

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)


/home/kenny/HTCV/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DLC 3.0.0rc13 | OpenCV 4.11.0


In [ ]:
SCENE       = 'scene2'      # the UNSEEN video we are testing
TRAIN_SCENE = 'scene1'      # the scene every model/calibration was built on
FPS         = 30

DATA_ROOT = Path('../data').resolve()

# scene2 (unseen) inputs
VAL_DIR   = DATA_ROOT / 'validation' / SCENE
STEM      = '2021-06-17_07-18-24_segment1_mouse291_combined'
RAW_TOP   = VAL_DIR / f'{STEM}_top-down-view.avi'
RAW_SIDE  = VAL_DIR / f'{STEM}_side-view.avi'
RAW_FRONT = VAL_DIR / f'{STEM}_front-view.avi'
GT_LABELS = VAL_DIR / f'{STEM}_labels.csv'

# scene1 trained artifacts we REUSE (nothing is retrained here) 
TRAIN_VAL_DIR = DATA_ROOT / 'validation' / TRAIN_SCENE
TRAIN_STEM    = '2021-06-18_07-28-45_segment1_mouse291_combined'
CALIB_TOML    = DATA_ROOT / 'lockbox_calibration' / TRAIN_SCENE / 'lockbox_calibration.toml'
SCHEMA_PATH   = DATA_ROOT / 'lockbox_dlc' / TRAIN_SCENE / 'state_schema.json'
PERVIEW_ROOT  = DATA_ROOT / 'lockbox_dlc' / TRAIN_SCENE / 'perview'
POSE_INPAINT  = DATA_ROOT / 'nn_output' / TRAIN_SCENE / 'pose_inpainter.pt'

# scene2 outputs 
MOUSE_OUT  = DATA_ROOT / 'multiview_output'   / SCENE
EXPORT_OUT = DATA_ROOT / 'pipeline_export'    / SCENE
TRIANG_OUT = DATA_ROOT / 'triangulate_render' / SCENE
NN_OUT     = DATA_ROOT / 'nn_output'          / SCENE
LOCKBOX_LB = DATA_ROOT / 'lockbox_dlc'        / SCENE / 'perview_infer'
STATE_OUT  = DATA_ROOT / 'lockbox_state'      / SCENE
EVAL_OUT   = DATA_ROOT / 'validation_eval'    / SCENE
for d in (MOUSE_OUT, EXPORT_OUT, TRIANG_OUT, NN_OUT, LOCKBOX_LB, STATE_OUT, EVAL_OUT):
    d.mkdir(parents=True, exist_ok=True)

print('Unseen scene videos:')
for p in (RAW_TOP, RAW_SIDE, RAW_FRONT, GT_LABELS):
    print(f'  {"OK " if p.exists() else "MISS":4s} {p.name}')
print('\nReused scene1 artifacts:')
for p in (CALIB_TOML, SCHEMA_PATH, PERVIEW_ROOT, POSE_INPAINT):
    print(f'  {"OK " if p.exists() else "MISS":4s} {p}')

Unseen scene videos:
  OK   2021-06-17_07-18-24_segment1_mouse291_combined_top-down-view.avi
  OK   2021-06-17_07-18-24_segment1_mouse291_combined_side-view.avi
  OK   2021-06-17_07-18-24_segment1_mouse291_combined_front-view.avi
  OK   2021-06-17_07-18-24_segment1_mouse291_combined_labels.csv

Reused scene1 artifacts:
  OK   /home/kenny/HTCV/data/lockbox_calibration/scene1/lockbox_calibration.toml
  OK   /home/kenny/HTCV/data/lockbox_dlc/scene1/state_schema.json
  OK   /home/kenny/HTCV/data/lockbox_dlc/scene1/perview
  OK   /home/kenny/HTCV/data/nn_output/scene1/pose_inpainter.pt


## 2. Mouse keypoints: run the *video1-adapted* SuperAnimal model on scene2

`video_inference_superanimal` re-runs video adaptation on every call. To test the
video1 model on an unseen video we instead pass the adapted snapshots from scene1
via `customized_pose_checkpoint` / `customized_detector_checkpoint` and set
`video_adapt=False`. Per view we reuse the same SuperAnimal head video1 used
(`topviewmouse` for top, `quadruped` for side/front).

Set `REUSE_ADAPTED_MODEL = False` to instead re-adapt on scene2 (treats scene2 as
its own session, a different experiment).

In [3]:
MOUSE_VIEWS = {
    'top':   dict(video=RAW_TOP,   sa='superanimal_topviewmouse', mb='hrnet_w32', src='top-down-view'),
    'side':  dict(video=RAW_SIDE,  sa='superanimal_quadruped',    mb='hrnet_w32', src='side-view'),
    'front': dict(video=RAW_FRONT, sa='superanimal_quadruped',    mb='hrnet_w32', src='front-view'),
}
REUSE_ADAPTED_MODEL = True

def adapted_ckpts(src):
    d = TRAIN_VAL_DIR / f'pseudo_{TRAIN_STEM}_{src}' / 'checkpoints'
    return d / 'snapshot-hrnet_w32-004.pt', d / 'snapshot-fasterrcnn_resnet50_fpn_v2-004.pt'

for v, cfg in MOUSE_VIEWS.items():
    pose, det = adapted_ckpts(cfg['src'])
    cfg['pose_ckpt'], cfg['det_ckpt'] = pose, det
    print(f'{v:5s} {cfg["sa"]:<26} pose={"OK" if pose.exists() else "MISS"}  det={"OK" if det.exists() else "MISS"}')

top   superanimal_topviewmouse   pose=OK  det=OK
side  superanimal_quadruped      pose=OK  det=OK
front superanimal_quadruped      pose=OK  det=OK


In [ ]:
INFER_KW = dict(
    detector_name='fasterrcnn_resnet50_fpn_v2',
    scale_list=[], batch_size=16, detector_batch_size=4,
    pcutoff=0.1, max_individuals=1, bbox_threshold=0.3,
    create_labeled_video=True, plot_bboxes=True,
)

def flush_vram():
    import torch; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

for v, cfg in MOUSE_VIEWS.items():
    out_dir = MOUSE_OUT / v; out_dir.mkdir(parents=True, exist_ok=True)
    done = [h for h in out_dir.glob('*.h5') if 'before_adapt' not in h.name]
    if done:
        print(f'skip {v}: {done[-1].name}'); continue
    if not cfg['video'].exists():
        print(f'skip {v}: video missing'); continue

    flush_vram()
    kw = dict(INFER_KW, video_adapt=False)
    if REUSE_ADAPTED_MODEL and cfg['pose_ckpt'].exists() and cfg['det_ckpt'].exists():
        kw.update(customized_pose_checkpoint=str(cfg['pose_ckpt']),
                  customized_detector_checkpoint=str(cfg['det_ckpt']))
        print(f'\n▶ {v}: video1-adapted weights on scene2 (no re-adaptation)')
    else:
        print(f'\n▶ {v}: stock {cfg["sa"]} weights (adapted ckpt not found / disabled)')

    video_inference_superanimal(
        videos=[str(cfg['video'])], superanimal_name=cfg['sa'], model_name=cfg['mb'],
        videotype=cfg['video'].suffix, dest_folder=str(out_dir), **kw)
    flush_vram()

print('\nMouse inference done')

skip top: 2021-06-17_07-18-24_segment1_mouse291_combined_top-down-view_superanimal_topviewmouse_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.h5
skip side: 2021-06-17_07-18-24_segment1_mouse291_combined_side-view_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.h5
skip front: 2021-06-17_07-18-24_segment1_mouse291_combined_front-view_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.h5

Mouse inference done


## 3. Load mouse predictions + keypoint mapping (same schema as Notebook 00)

In [5]:
def load_predictions(view):
    folder = MOUSE_OUT / view
    h5s = [h for h in folder.glob('*.h5') if 'before_adapt' not in h.name]
    if not h5s: return None
    df = pd.read_hdf(sorted(h5s)[-1])
    while df.columns.nlevels > 2:
        df = df.droplevel(0, axis=1)
    return df

all_preds = {v: load_predictions(v) for v in MOUSE_VIEWS}
for v, df in all_preds.items():
    s = 'none' if df is None else f'{len(df)} frames, {len(df.columns.get_level_values(0).unique())} kps'
    print(f'  {v:5s}: {s}')
assert all(df is not None for df in all_preds.values()), 'a view has no predictions — re-run section 2'

  top  : 5670 frames, 27 kps
  side : 5670 frames, 39 kps
  front: 5670 frames, 39 kps


In [6]:
VIEWS_3D       = {'top': 'top', 'side': 'side', 'front': 'front'}
VIDEO_FOR_VIEW = {'top': RAW_TOP, 'side': RAW_SIDE, 'front': RAW_FRONT}
bp_by_view = {v: all_preds[v].columns.get_level_values(0).unique().tolist() for v in ['top', 'side', 'front']}

KEYPOINT_MAP = {
    # canonical          top                 side                 front
    'nose':              ('nose',            'nose',              'nose'),
    'left_eye':          ('left_eye',        'left_eye',          'left_eye'),
    'right_eye':         ('right_eye',       'right_eye',         'right_eye'),
    'left_ear':          ('left_ear',        'left_earbase',      'left_earbase'),
    'right_ear':         ('right_ear',       'right_earbase',     'right_earbase'),
    'left_ear_tip':      ('left_ear_tip',    'left_earend',       'left_earend'),
    'right_ear_tip':     ('right_ear_tip',   'right_earend',      'right_earend'),
    'head_mid':          ('head_midpoint',   'neck_end',          'neck_end'),
    'neck':              ('neck',            'neck_base',         'neck_base'),
    'mid_back':          ('mid_back',        'back_middle',       'back_middle'),
    'tail_base':         ('tail_base',       'tail_base',         'tail_base'),
    'tail_end':          ('tail_end',        'tail_end',          'tail_end'),
    'left_shoulder':     ('left_shoulder',   'front_left_thai',   'front_left_thai'),
    'right_shoulder':    ('right_shoulder',  'front_right_thai',  'front_right_thai'),
    'left_hip':          ('left_hip',        'back_left_thai',    'back_left_thai'),
    'right_hip':         ('right_hip',       'back_right_thai',   'back_right_thai'),
    'front_left_paw':    (None,              'front_left_paw',    'front_left_paw'),
    'front_right_paw':   (None,              'front_right_paw',   'front_right_paw'),
    'back_left_paw':     (None,              'back_left_paw',     'back_left_paw'),
    'back_right_paw':    (None,              'back_right_paw',    'back_right_paw'),
}

valid_kps = []
for canon, (t, s, f) in KEYPOINT_MAP.items():
    ok_t = (t is None) or (t in bp_by_view['top'])
    ok_s = (s is None) or (s in bp_by_view['side'])
    ok_f = (f is None) or (f in bp_by_view['front'])
    if ok_t and ok_s and ok_f:
        valid_kps.append(canon)
n_kps = len(valid_kps)

SKELETON = [
    ('nose', 'left_eye'),  ('nose', 'right_eye'),
    ('left_eye', 'left_ear'), ('right_eye', 'right_ear'),
    ('left_ear', 'left_ear_tip'), ('right_ear', 'right_ear_tip'),
    ('left_ear', 'head_mid'), ('right_ear', 'head_mid'),
    ('head_mid', 'neck'),  ('neck', 'mid_back'),
    ('mid_back', 'tail_base'), ('tail_base', 'tail_end'),
    ('neck', 'left_shoulder'),  ('neck', 'right_shoulder'),
    ('tail_base', 'left_hip'),  ('tail_base', 'right_hip'),
    ('left_shoulder', 'front_left_paw'), ('right_shoulder', 'front_right_paw'),
    ('left_hip', 'back_left_paw'), ('right_hip', 'back_right_paw'),
]

with open(EXPORT_OUT / '2d_state.pkl', 'wb') as f:
    pickle.dump(dict(
        view_predictions={v: all_preds[v] for v in ['top', 'side', 'front']},
        valid_kps=valid_kps, KEYPOINT_MAP=KEYPOINT_MAP, SKELETON=SKELETON,
        VIEWS_3D=VIEWS_3D, VIDEO_FOR_VIEW={k: str(v) for k, v in VIDEO_FOR_VIEW.items()},
        n_kps=n_kps), f)
print(f'{n_kps} valid keypoints -> {EXPORT_OUT / "2d_state.pkl"}')

20 valid keypoints -> /home/kenny/HTCV/data/pipeline_export/scene2/2d_state.pkl


## 4. Triangulate the mouse with the **video1 calibration**

Same `CameraGroup` as scene1, same multi-pass single-view fallback and short-gap
fill as Notebook 02. Output mirrors `triangulate_render/<scene>/pipeline_state.pkl`.

In [7]:
import collections
cgroup     = CameraGroup.load(str(CALIB_TOML))
view_order = ['top', 'side', 'front']
n_cams     = 3
CONF       = 0.5
n_frames   = min(len(all_preds[v]) for v in view_order)

points_2d = np.full((n_cams, n_frames, n_kps, 2), np.nan)
for c, view in enumerate(view_order):
    df = all_preds[view]
    for k, canon in enumerate(valid_kps):
        bp = KEYPOINT_MAP[canon][c]
        if bp is None or bp not in df.columns.get_level_values(0): continue
        x, y, l = df[bp]['x'].values[:n_frames], df[bp]['y'].values[:n_frames], df[bp]['likelihood'].values[:n_frames]
        m = l > CONF
        points_2d[c, m, k, 0] = x[m]; points_2d[c, m, k, 1] = y[m]

points_3d    = np.full((n_frames, n_kps, 3), np.nan)
n_views_used = np.zeros((n_frames, n_kps), np.int8)
for fr in range(n_frames):
    fp = points_2d[:, fr]
    n_views_used[fr] = (~np.isnan(fp[..., 0])).sum(axis=0)
    mask = n_views_used[fr] >= 2
    if mask.any():
        points_3d[fr, mask] = cgroup.triangulate(fp[:, mask, :], undistort=True)
print(f'triangulated coverage (>=2 views): {np.isfinite(points_3d).all(-1).mean()*100:.1f}%')

triangulated coverage (>=2 views): 14.9%


In [8]:
# Multi-pass single-view fallback + temporal anchor (Notebook 02 logic)
FALLBACK_REF = {
    'front_left_paw':'left_shoulder','front_right_paw':'right_shoulder',
    'back_left_paw':'left_hip','back_right_paw':'right_hip',
    'left_shoulder':'neck','right_shoulder':'neck',
    'left_hip':'tail_base','right_hip':'tail_base',
}
def cam_center(cam):
    R,_ = cv2.Rodrigues(cam.rvec); return (-R.T @ cam.tvec.reshape(3,1)).ravel()
def back_ray(cam, pt2d):
    pt = np.array([pt2d[0], pt2d[1], 1.0]); R,_ = cv2.Rodrigues(cam.rvec)
    d = R.T @ (np.linalg.inv(cam.matrix) @ pt); return cam_center(cam), d/np.linalg.norm(d)
def snap_to_anchor(cam, pt2d, anchor):
    C,d = back_ray(cam, pt2d); return C + np.dot(anchor - C, d)*d

SKEL_ADJ = {}
for a,b in SKELETON:
    SKEL_ADJ.setdefault(a,set()).add(b); SKEL_ADJ.setdefault(b,set()).add(a)
def find_anchor(canon, frame_3d):
    ref = FALLBACK_REF.get(canon)
    if ref in valid_kps and np.isfinite(frame_3d[valid_kps.index(ref)]).all():
        return frame_3d[valid_kps.index(ref)], 'fallback_ref'
    for nb in SKEL_ADJ.get(canon, []):
        if nb in valid_kps and np.isfinite(frame_3d[valid_kps.index(nb)]).all():
            return frame_3d[valid_kps.index(nb)], 'skeleton'
    m = np.isfinite(frame_3d).all(axis=1)
    if m.any(): return frame_3d[m].mean(axis=0), 'centroid'
    return None, None

single_view_mask = np.zeros((n_frames, n_kps), bool)
anchor_source    = np.empty((n_frames, n_kps), object); anchor_source[:] = None
for fr in range(n_frames):
    for _ in range(3):
        added = False
        for k, canon in enumerate(valid_kps):
            if np.isfinite(points_3d[fr,k]).all() or n_views_used[fr,k] != 1: continue
            c = int(np.where(~np.isnan(points_2d[:,fr,k,0]))[0][0])
            anchor, src = find_anchor(canon, points_3d[fr])
            if anchor is None: continue
            points_3d[fr,k] = snap_to_anchor(cgroup.cameras[c], points_2d[c,fr,k], anchor)
            single_view_mask[fr,k] = True; anchor_source[fr,k] = src; added = True
        if not added: break

TEMPORAL_WINDOW = 30
for fr in range(n_frames):
    for k, canon in enumerate(valid_kps):
        if np.isfinite(points_3d[fr,k]).all() or n_views_used[fr,k] != 1: continue
        c = int(np.where(~np.isnan(points_2d[:,fr,k,0]))[0][0]); anchor=None
        for pf in range(fr-1, max(-1, fr-TEMPORAL_WINDOW), -1):
            if np.isfinite(points_3d[pf,k]).all(): anchor=points_3d[pf,k]; break
        if anchor is None:
            for nf in range(fr+1, min(n_frames, fr+TEMPORAL_WINDOW)):
                if np.isfinite(points_3d[nf,k]).all(): anchor=points_3d[nf,k]; break
        if anchor is None: continue
        points_3d[fr,k]=snap_to_anchor(cgroup.cameras[c], points_2d[c,fr,k], anchor)
        single_view_mask[fr,k]=True; anchor_source[fr,k]='temporal'

def fill_short_gaps(arr, max_gap=5):
    out = arr.copy(); n_f,n_k,_ = out.shape
    for k in range(n_k):
        for c in range(3):
            s = out[:,k,c]; valid = np.isfinite(s)
            if valid.sum() < 2: continue
            inv = ~valid; diff = np.diff(inv.astype(int))
            starts = np.where(diff==1)[0]+1; ends = np.where(diff==-1)[0]+1
            if inv[0]: starts=np.concatenate([[0],starts])
            if inv[-1]: ends=np.concatenate([ends,[n_f]])
            for gs,ge in zip(starts,ends):
                if (ge-gs)>max_gap or gs==0 or ge==n_f: continue
                y0,y1=s[gs-1],s[ge]
                for fi in range(gs,ge): out[fi,k,c]=y0+(y1-y0)*(fi-(gs-1))/(ge-(gs-1))
    return out

before = np.isfinite(points_3d).all(-1)
points_3d = fill_short_gaps(points_3d, 5)
CRITICAL = {'nose','tail_base','neck','mid_back'}
crit = [valid_kps.index(k) for k in CRITICAL if k in valid_kps]
if crit:
    big = fill_short_gaps(points_3d, 30)
    for k in crit: points_3d[:,k,:] = big[:,k,:]
interp_mask = np.isfinite(points_3d).all(-1) & ~before

with open(TRIANG_OUT / 'pipeline_state.pkl', 'wb') as f:
    pickle.dump(dict(points_3d=points_3d, n_views_used=n_views_used,
        single_view_mask=single_view_mask, anchor_source=anchor_source, interp_mask=interp_mask,
        valid_kps=valid_kps, SKELETON=SKELETON, VIEWS_3D=VIEWS_3D,
        VIDEO_FOR_VIEW={k:str(v) for k,v in VIDEO_FOR_VIEW.items()},
        view_predictions={v:all_preds[v] for v in view_order}, KEYPOINT_MAP=KEYPOINT_MAP,
        n_frames=n_frames, n_kps=n_kps, calibration_source='lockbox_pnp_scene1'), f)
print(f'mouse 3D coverage after fallback+fill: {np.isfinite(points_3d).all(-1).mean()*100:.1f}%')
print(f'{TRIANG_OUT / "pipeline_state.pkl"}')

mouse 3D coverage after fallback+fill: 61.3%
/home/kenny/HTCV/data/triangulate_render/scene2/pipeline_state.pkl


## 5. (optional) Fill mouse 3D with the **video1 pose-inpainter**

Loads `pose_inpainter.pt` from scene1 and applies it to scene2's triangulated 3D
(same windowed transformer as Notebook 03). Skipped automatically if the checkpoint
is missing or its `n_kps` differs from scene2's.

In [9]:
import torch, torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def interpolate_3d(arr):
    out = arr.copy(); n_f,n_k,_ = out.shape
    for k in range(n_k):
        for c in range(3):
            s = out[:,k,c]; v = np.isfinite(s)
            if v.sum() >= 2: out[:,k,c] = np.interp(np.arange(n_f), np.where(v)[0], s[v])
    return out

points_3d_nn = points_3d.copy()
run_nn = POSE_INPAINT.exists()
if run_nn:
    ckpt = torch.load(POSE_INPAINT, map_location=device, weights_only=False)
    cfg = ckpt['config']; mean_p = ckpt['mean_p']; std_p = ckpt['std_p']
    if cfg['n_kps'] != n_kps:
        print(f'skip NN: checkpoint n_kps={cfg["n_kps"]} != scene2 n_kps={n_kps}'); run_nn = False

if run_nn:
    WINDOW = cfg['window']; HALF = WINDOW // 2
    class PoseInpainterT(nn.Module):
        def __init__(s, n_kps, window, d_model, nhead, n_layers, dropout=0.1):
            super().__init__(); s.n_kps, s.window = n_kps, window
            s.input_proj = nn.Linear(4, d_model)
            s.kp_embed = nn.Embedding(n_kps, d_model); s.frame_embed = nn.Embedding(window, d_model)
            enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
                    dropout=dropout, batch_first=True, activation='gelu', norm_first=True)
            s.encoder = nn.TransformerEncoder(enc, num_layers=n_layers); s.output_proj = nn.Linear(d_model, 3)
        def forward(s, pose, mask):
            B,T,K,_ = pose.shape
            x = torch.cat([pose*mask.unsqueeze(-1), mask.unsqueeze(-1)], dim=-1)
            x = s.input_proj(x)
            x = x + s.kp_embed(torch.arange(K, device=x.device))[None,None]
            x = x + s.frame_embed(torch.arange(T, device=x.device))[None,:,None]
            x = s.encoder(x.reshape(B, T*K, -1)).reshape(B,T,K,-1)
            return s.output_proj(x)
    model = PoseInpainterT(cfg['n_kps'], cfg['window'], cfg['d_model'], cfg['nhead'], cfg['n_layers']).to(device)
    model.load_state_dict(ckpt['state_dict']); model.eval()

    points_3d_interp = interpolate_3d(points_3d)
    pad = np.concatenate([np.repeat(points_3d[:1],HALF,0), points_3d, np.repeat(points_3d[-1:],HALF,0)], 0)
    nn_filled = np.zeros((n_frames, n_kps), bool)
    with torch.no_grad():
        for fr in range(n_frames):
            tv = np.isfinite(points_3d[fr]).all(axis=1)
            if tv.all(): continue
            if tv.sum() < 3:
                points_3d_nn[fr][~tv] = points_3d_interp[fr][~tv]; nn_filled[fr] = ~tv; continue
            win = pad[fr:fr+WINDOW]; in_mask = np.isfinite(win).all(-1).astype(np.float32)
            x = ((np.where(np.isfinite(win), win, 0.0) - mean_p)/std_p).astype(np.float32)*in_mask[...,None]
            pred = model(torch.from_numpy(x[None]).to(device), torch.from_numpy(in_mask[None]).to(device))
            cp = pred.cpu().numpy()[0][HALF]*std_p + mean_p
            for k in range(n_kps):
                if not tv[k]: points_3d_nn[fr,k]=cp[k]; nn_filled[fr,k]=True
    print(f'NN coverage: {np.isfinite(points_3d_nn).all(-1).mean()*100:.1f}%  (filled {int(nn_filled.sum())})')

    rows=[]
    for fr in range(n_frames):
        for k,name in enumerate(valid_kps):
            if not np.isfinite(points_3d_nn[fr,k]).all(): continue
            rows.append(dict(frame=fr, keypoint=name, source='nn',
                x=points_3d_nn[fr,k,0], y=points_3d_nn[fr,k,1], z=points_3d_nn[fr,k,2],
                nn_filled=bool(nn_filled[fr,k])))
    pd.DataFrame(rows).to_csv(NN_OUT / 'points_3d_all_sources.csv', index=False)
    print(f' {NN_OUT / "points_3d_all_sources.csv"}')
else:
    print('NN inpainting skipped.')

NN coverage: 100.0%  (filled 43844)


 /home/kenny/HTCV/data/nn_output/scene2/points_3d_all_sources.csv


## 6. Lockbox keypoints: run the **video1-trained per-view DLC models** on scene2

`analyze_videos` with each scene1 per-view project, pointed at the scene2 video.
Outputs land in `lockbox_dlc/scene2/perview_infer/<view>/`.

In [10]:
import deeplabcut as dlc, yaml
SCORER, SHUFFLE = 'htcv', 1
dlc_device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

def perview_config(view):
    h = sorted(PERVIEW_ROOT.glob(f'lockbox_{view}-{SCORER}-*/config.yaml'))
    return h[-1] if h else None

LB_VIDEO = {'top': RAW_TOP, 'side': RAW_SIDE, 'front': RAW_FRONT}
LB_H5 = {}
for v, vid in LB_VIDEO.items():
    cfgp = perview_config(v); assert cfgp, f'no per-view project for {v}'
    dest = LOCKBOX_LB / v; dest.mkdir(parents=True, exist_ok=True)
    existing = [h for h in dest.glob('*.h5') if 'filtered' not in h.name.lower()]
    if existing:
        LB_H5[v] = sorted(existing)[-1]; print(f'skip {v}: {LB_H5[v].name}'); continue
    cfg = yaml.safe_load(open(cfgp)); cfg['snapshotindex'] = -1
    yaml.safe_dump(cfg, open(cfgp, 'w'), sort_keys=False)
    print(f'▶ analyze {v}  ({cfgp.parent.name})')
    dlc.analyze_videos(str(cfgp), [str(vid)], shuffle=SHUFFLE, device=dlc_device,
                       destfolder=str(dest), save_as_csv=False)
    LB_H5[v] = sorted(h for h in dest.glob('*.h5') if 'filtered' not in h.name.lower())[-1]
    print(f'  -> {LB_H5[v].name}')
print('\n lockbox inference done')

skip top: 2021-06-17_07-18-24_segment1_mouse291_combined_top-down-viewDLC_Resnet50_lockbox_topMay25shuffle1_snapshot_best-120.h5
skip side: 2021-06-17_07-18-24_segment1_mouse291_combined_side-viewDLC_Resnet50_lockbox_sideMay25shuffle1_snapshot_best-130.h5
skip front: 2021-06-17_07-18-24_segment1_mouse291_combined_front-viewDLC_Resnet50_lockbox_frontMay25shuffle1_snapshot_best-120.h5

 lockbox inference done


## 7. Lockbox state extraction (same schema + calibration as scene1)

Notebook 06 logic: load schema, merge the per-view lockbox predictions, triangulate
with the shared `CameraGroup`, single-view fallback + gap fill, continuous state per
mechanism, then the sequential discrete stage machine. Saves
`lockbox_state/scene2/lockbox_state.{csv,pkl}`.

In [11]:
with open(SCHEMA_PATH) as f:
    LOCKBOX_SCHEMA = json.load(f)

MECHANISMS, STATE_RANGE, RANGE_BOX = {}, {}, {}
MOVING_KPS, STATIC_KPS = [], []
for m in LOCKBOX_SCHEMA['mechanisms']:
    name, t = m['name'], m['type']; mv = m['state_keypoint']; refs = list(m.get('reference_keypoints', []))
    MOVING_KPS.append(mv); STATIC_KPS.extend(refs)
    e = dict(type=t, moving=mv, refs=refs, extraction=m.get('state_extraction'))
    if t in ('revolute','prismatic'):
        d = np.asarray(m['axis_direction_lockbox'], float)
        e['axis_origin'] = np.asarray(m['axis_origin_lockbox'], float); e['axis_dir'] = d/np.linalg.norm(d)
        if t == 'revolute':
            e['unit']='rad'; STATE_RANGE[name]=tuple(m['range_rad']); e['extraction']=e['extraction'] or 'angle_around_axis'
            if 'rest_direction_lockbox' in m:
                r=np.asarray(m['rest_direction_lockbox'],float); e['rest_dir']=r/np.linalg.norm(r)
            e['preferred_view']=m.get('preferred_view')
        else:
            e['unit']='mm'; STATE_RANGE[name]=tuple(m['range_mm']); e['extraction']=e['extraction'] or 'project_on_axis'
    elif t == 'free':
        e['unit']='mm'; e['rest_pos']=np.asarray(m['rest_position_lockbox'],float)
        e['extraction']=e['extraction'] or 'distance_from_rest'; e['removed_radius_mm']=float(m.get('removed_radius_mm',20.0))
        RANGE_BOX[name]=m.get('range_mm_box')
    MECHANISMS[name]=e
BODYPARTS = sorted(set(MOVING_KPS) | set(STATIC_KPS))

ENGAGE = {'lever1':('above', np.deg2rad(45)), 'slider1':('above',5.0),
          'ball1':('above', MECHANISMS['ball1']['removed_radius_mm']), 'cover1':('above',10.0)}
STAGE_ORDER = ['lever1','slider1','ball1','cover1']
STAGE_NAMES = ['start','lever1_pivoted','slider1_slid','ball_removed','cover_slid']
MIN_PERSIST = 5
LB_CONF = 0.5
print('mechanisms:', list(MECHANISMS))

mechanisms: ['lever1', 'slider1', 'ball1', 'cover1']


In [12]:
def load_dlc_h5(path):
    df = pd.read_hdf(path)
    while df.columns.nlevels > 2: df = df.droplevel(0, axis=1)
    return df

lb_view_order = [c.get_name() for c in cgroup.cameras]
lb_pred = {v: load_dlc_h5(LB_H5[v]) for v in lb_view_order}
lb_nf   = min(len(df) for df in lb_pred.values())
present = set().union(*[set(df.columns.get_level_values(0)) for df in lb_pred.values()])
lb_valid = [bp for bp in BODYPARTS if bp in present]
lb_nk = len(lb_valid)

lb_2d = np.full((3, lb_nf, lb_nk, 2), np.nan)
lb_lik = np.zeros((3, lb_nf, lb_nk), np.float32)
for c, v in enumerate(lb_view_order):
    df = lb_pred[v]; cols = set(df.columns.get_level_values(0))
    for k, bp in enumerate(lb_valid):
        if bp not in cols: continue
        x,y,l = df[bp]['x'].values[:lb_nf], df[bp]['y'].values[:lb_nf], df[bp]['likelihood'].values[:lb_nf]
        mm = l > LB_CONF
        lb_2d[c,mm,k,0]=x[mm]; lb_2d[c,mm,k,1]=y[mm]; lb_lik[c,:,k]=l

lb_3d = np.full((lb_nf, lb_nk, 3), np.nan)
lb_nviews = np.zeros((lb_nf, lb_nk), np.int8)
for fr in range(lb_nf):
    fp = lb_2d[:,fr]; lb_nviews[fr] = (~np.isnan(fp[...,0])).sum(axis=0)
    mask = lb_nviews[fr] >= 2
    if mask.any(): lb_3d[fr,mask] = cgroup.triangulate(fp[:,mask,:], undistort=True)
print(f'lockbox keypoints (union): {lb_nk}/{len(BODYPARTS)} | coverage {np.isfinite(lb_3d).all(-1).mean()*100:.1f}%')

lockbox keypoints (union): 8/8 | coverage 46.9%


In [13]:
# single-view fallback + short-gap fill (Notebook 06)
def _median_point(bp):
    if bp not in lb_valid: return None
    col = lb_3d[:, lb_valid.index(bp), :]; col = col[np.isfinite(col).all(axis=1)]
    return np.median(col, axis=0) if len(col) else None

FALLBACK_REF_LB = {m['moving']:(m['refs'][0] if m['refs'] else None) for m in MECHANISMS.values()}
static_median = {bp: _median_point(bp) for bp in STATIC_KPS if _median_point(bp) is not None}
lb_single = np.zeros((lb_nf, lb_nk), bool)
for fr in range(lb_nf):
    for k, bp in enumerate(lb_valid):
        if np.isfinite(lb_3d[fr,k]).all() or lb_nviews[fr,k] != 1: continue
        ref = FALLBACK_REF_LB.get(bp); anchor = static_median.get(ref) if ref in static_median else None
        if anchor is None and ref in lb_valid and np.isfinite(lb_3d[fr,lb_valid.index(ref)]).all():
            anchor = lb_3d[fr, lb_valid.index(ref)]
        if anchor is None: continue
        c = int(np.where(~np.isnan(lb_2d[:,fr,k,0]))[0][0])
        lb_3d[fr,k] = snap_to_anchor(cgroup.cameras[c], lb_2d[c,fr,k], anchor); lb_single[fr,k]=True

before = np.isfinite(lb_3d).all(-1)
lb_3d = fill_short_gaps(lb_3d, 5)
lb_interp = np.isfinite(lb_3d).all(-1) & ~before
print(f'lockbox coverage after fallback+fill: {np.isfinite(lb_3d).all(-1).mean()*100:.1f}%')

lockbox coverage after fallback+fill: 65.1%


In [14]:
# continuous state per mechanism (Notebook 06)
def _track(bp): return lb_3d[:, lb_valid.index(bp), :]
def _nv(bp):    return lb_nviews[:, lb_valid.index(bp)]
def project_on_axis(track, origin, axis): return (track-origin) @ axis
def plane_basis(axis):
    W=np.eye(3); ref=W[int(np.argmin(np.abs(W@axis)))]
    e1=ref-(ref@axis)*axis; e1/=np.linalg.norm(e1); return e1, np.cross(axis,e1)
def angle_around_axis(track, origin, axis):
    v=track-origin; e1,e2=plane_basis(axis); a=np.arctan2(v@e2, v@e1)
    a[~np.isfinite(track).all(axis=1)]=np.nan; return a
def lever_angle_top(mech):
    cam_idx = lb_view_order.index(mech['preferred_view']); cam = cgroup.cameras[cam_idx]
    axis_dir = mech['axis_dir']; rest_dir = mech.get('rest_dir')
    if rest_dir is None: rest_dir,_ = plane_basis(axis_dir)
    perp = np.cross(axis_dir, rest_dir); perp/=np.linalg.norm(perp)
    pivot = mech['axis_origin']; k = lb_valid.index(mech['moving'])
    pivot_2d = cam.project(pivot.reshape(1,3))[0]
    d_2d = np.linalg.norm(lb_2d[cam_idx,:,k,:]-pivot_2d, axis=1)
    det = np.isfinite(d_2d).astype(np.int8)
    if det.sum() < 10: return np.full(lb_nf,np.nan), det
    tr = lb_3d[:,k,:]; fin = np.isfinite(tr).all(axis=1)
    L3 = float(np.median(np.linalg.norm(tr[fin]-pivot, axis=1))) if fin.sum()>=10 else None
    if L3 and L3>0.5:
        ref2 = cam.project(np.vstack([pivot, pivot+L3*perp])); L2 = float(np.linalg.norm(ref2[1]-ref2[0]))
    else:
        L2 = float(np.nanpercentile(d_2d, 99))
    if not np.isfinite(L2) or L2<1.0: return np.full(lb_nf,np.nan), det
    a = np.arcsin(np.clip(d_2d/L2, 0, 1)); a[~np.isfinite(d_2d)] = np.nan
    return a, det

state = {}
for name, m in MECHANISMS.items():
    if m['moving'] not in lb_valid: print(f'skip {name}: {m["moving"]} not predicted'); continue
    ext = m['extraction']; nv = _nv(m['moving'])
    if ext == 'project_on_axis':
        state[name]=dict(value=project_on_axis(_track(m['moving']), m['axis_origin'], m['axis_dir']),
                         nviews=nv, unit=m['unit'], kind='prismatic')
    elif ext == 'angle_around_axis':
        state[name]=dict(value=angle_around_axis(_track(m['moving']), m['axis_origin'], m['axis_dir']),
                         nviews=nv, unit=m['unit'], kind='revolute')
    elif ext in ('angle_relative_top_view','angle_top_apparent_length'):
        v,det = lever_angle_top(m); state[name]=dict(value=v, nviews=det, unit=m['unit'], kind='revolute_top')
    elif ext in ('identity_3d','distance_from_rest'):
        tr=_track(m['moving']); state[name]=dict(value=np.linalg.norm(tr-m['rest_pos'],axis=1),
                         xyz=tr, nviews=nv, unit=m['unit'], kind='free')

# range filter
for name, st in state.items():
    if st['kind']=='free': continue
    lo_r,hi_r = STATE_RANGE.get(name,(-np.inf,np.inf)); v=st['value']
    v[np.isfinite(v) & ((v<lo_r)|(v>hi_r))] = np.nan; st['value']=v
print('continuous state extracted:', {n: round(float(np.nanmax(s['value'])),2) for n,s in state.items()})

continuous state extracted: {'lever1': 1.57, 'slider1': 39.52, 'ball1': 25.32, 'cover1': 22.11}


In [15]:
# discrete sequential stage machine (Notebook 06)
def engaged_mask(name):
    v = state[name]['value']; op,thr = ENGAGE[name]
    e = {'below':v<thr,'above':v>thr,'abs_above':np.abs(v)>thr,'abs_below':np.abs(v)<thr}[op]
    return np.where(np.isfinite(v), e, False)
engaged = {n: engaged_mask(n) for n in STAGE_ORDER if n in state}
def first_sustained(b, k, start=0):
    cnt=0
    for i in range(start, len(b)):
        cnt = cnt+1 if b[i] else 0
        if cnt>=k: return i-k+1
    return None
onsets, prev = {}, 0
for name in STAGE_ORDER:
    o = first_sustained(engaged[name], MIN_PERSIST, prev) if name in engaged else None
    onsets[name]=o
    if o is not None: prev=o
stage = np.zeros(lb_nf, int)
for idx,name in enumerate(STAGE_ORDER, 1):
    if onsets.get(name) is not None: stage[onsets[name]:]=idx
print('PREDICTED stage onsets on scene2:')
for idx,name in enumerate(STAGE_ORDER,1):
    o=onsets.get(name); print(f'  {idx}. {STAGE_NAMES[idx]:<16} -> {"frame %d (%.1fs)"%(o,o/FPS) if o is not None else "NOT reached"}')

out = {'frame':np.arange(lb_nf), 'time_s':np.arange(lb_nf)/FPS,
       'stage':stage, 'stage_name':[STAGE_NAMES[s] for s in stage]}
for name, st in state.items():
    out[name]=st['value']; out[f'{name}_nviews']=st['nviews']
    if name in engaged: out[f'{name}_engaged']=engaged[name]
    if st['kind']=='free':
        out[f'{name}_x'],out[f'{name}_y'],out[f'{name}_z']=st['xyz'][:,0],st['xyz'][:,1],st['xyz'][:,2]
df_state = pd.DataFrame(out); df_state.to_csv(STATE_OUT/'lockbox_state.csv', index=False)
with open(STATE_OUT/'lockbox_state.pkl','wb') as f:
    pickle.dump(dict(points_2d=lb_2d, points_3d=lb_3d, likelihoods=lb_lik, n_views_used=lb_nviews,
        single_view_mask=lb_single, interp_mask=lb_interp, valid_kps=lb_valid, view_order=lb_view_order,
        n_frames=lb_nf, n_kps=lb_nk, MECHANISMS=MECHANISMS, STATE_RANGE=STATE_RANGE, RANGE_BOX=RANGE_BOX,
        state={k:dict(v) for k,v in state.items()}, engaged=engaged, stage=stage, onsets=onsets,
        STAGE_ORDER=STAGE_ORDER, STAGE_NAMES=STAGE_NAMES, ENGAGE=ENGAGE,
        VIDEO_FOR_VIEW={k:str(v) for k,v in LB_VIDEO.items()},
        calibration_source='lockbox_pnp_scene1'), f)
print(f' {STATE_OUT/"lockbox_state.csv"}')

PREDICTED stage onsets on scene2:
  1. lever1_pivoted   -> frame 2089 (69.6s)
  2. slider1_slid     -> frame 2109 (70.3s)
  3. ball_removed     -> NOT reached
  4. cover_slid       -> frame 2902 (96.7s)
 /home/kenny/HTCV/data/lockbox_state/scene2/lockbox_state.csv


## 8. Evaluate predictions against scene2's ground-truth `labels.csv`

scene2 ships per-frame ground-truth state columns. We map each pipeline mechanism
to its GT column, build the GT discrete stage the same sequential way, and compare
onsets + per-frame stage agreement. **This is the actual generalization number.**

Mechanism to GT column mapping (edit if your lockbox differs):
`lever1 to lever_state`, `slider1 to stick_state`, `ball1 to ball_state`, `cover1 to sliding_door_state`.

In [16]:
gt = pd.read_csv(GT_LABELS)
GT_STATE_BASE = {'lever1':'lever_state','slider1':'stick_state','ball1':'ball_state','cover1':'sliding_door_state'}

def pick_arena(base):
    a1, a2 = f'{base}_A1', f'{base}_A2'
    if a1 in gt and a2 in gt:
        return a1 if np.nanstd(gt[a1].values) >= np.nanstd(gt[a2].values) else a2
    return a1 if a1 in gt else a2

gt_cols = {m: pick_arena(b) for m, b in GT_STATE_BASE.items()}
N = min(lb_nf, len(gt))
print('chosen GT columns (more-varying arena):')
for m, c in gt_cols.items():
    print(f'  {m:<8} <- {c:<22} range [{gt[c][:N].min():.2f}, {gt[c][:N].max():.2f}]')

# GT engaged: state column departs from its at-rest (first-frame) value
gt_engaged = {}
for m, c in gt_cols.items():
    col = gt[c].values[:N].astype(float)
    rest = col[0]
    gt_engaged[m] = np.abs(col - rest) > 1e-6

# GT sequential stage (same debounce + ordering as the prediction)
gt_onsets, prev = {}, 0
for name in STAGE_ORDER:
    o = first_sustained(gt_engaged[name], MIN_PERSIST, prev); gt_onsets[name]=o
    if o is not None: prev=o
gt_stage = np.zeros(N, int)
for idx, name in enumerate(STAGE_ORDER, 1):
    if gt_onsets.get(name) is not None: gt_stage[gt_onsets[name]:]=idx

pred_stage = stage[:N]
print('\n         predicted onset      ground-truth onset     |Δ|')
rows=[]
for idx, name in enumerate(STAGE_ORDER, 1):
    po, go = onsets.get(name), gt_onsets.get(name)
    ps = f'{po/FPS:6.1f}s' if po is not None else '  --  '
    gs = f'{go/FPS:6.1f}s' if go is not None else '  --  '
    derr = abs(po-go)/FPS if (po is not None and go is not None) else np.nan
    rows.append(dict(mechanism=name, stage=STAGE_NAMES[idx], pred_onset_s=po/FPS if po is not None else None,
                     gt_onset_s=go/FPS if go is not None else None, abs_err_s=derr))
    print(f'  {STAGE_NAMES[idx]:<16} {ps}            {gs}        {derr:5.2f}s' if np.isfinite(derr)
          else f'  {STAGE_NAMES[idx]:<16} {ps}            {gs}          --')
eval_df = pd.DataFrame(rows); eval_df.to_csv(EVAL_OUT/'onset_eval.csv', index=False)

acc = (pred_stage == gt_stage).mean()
mae = np.abs(pred_stage - gt_stage).mean()
print(f'\nper-frame stage accuracy : {acc*100:.1f}%')
print(f'per-frame stage MAE      : {mae:.3f}')
print(f'reached stage (pred/gt)  : {int(pred_stage.max())} / {int(gt_stage.max())}')

chosen GT columns (more-varying arena):
  lever1   <- lever_state_A1         range [0.00, 1.00]
  slider1  <- stick_state_A1         range [0.00, 1.00]
  ball1    <- ball_state_A2          range [0.00, 1.00]
  cover1   <- sliding_door_state_A1  range [0.00, 1.00]

         predicted onset      ground-truth onset     |Δ|
  lever1_pivoted     69.6s              45.0s        24.67s
  slider1_slid       70.3s              81.3s        11.00s
  ball_removed       --               114.0s          --
  cover_slid         96.7s             136.2s        39.47s



per-frame stage accuracy : 60.2%
per-frame stage MAE      : 0.489
reached stage (pred/gt)  : 4 / 4


In [17]:
# Timeline: predicted vs ground-truth stage + per-mechanism engagement
t = np.arange(N)/FPS
fig, axes = plt.subplots(len(STAGE_ORDER)+1, 1, figsize=(13, 2.0*(len(STAGE_ORDER)+1)), sharex=True)

for ax, name in zip(axes[:-1], STAGE_ORDER):
    if name in state:
        ax.plot(t, state[name]['value'][:N], lw=0.7, color='0.5')
        if name in ENGAGE: ax.axhline(ENGAGE[name][1], ls='--', c='purple', lw=0.8)
    twin = ax.twinx()
    twin.fill_between(t, 0, gt_engaged[name].astype(int), color='tab:green', alpha=0.20, step='post', label='GT engaged')
    twin.set_yticks([]); twin.set_ylim(0,1.05)
    if onsets.get(name) is not None: ax.axvline(onsets[name]/FPS, color='tab:blue', lw=1.2, label='pred onset')
    if gt_onsets.get(name) is not None: ax.axvline(gt_onsets[name]/FPS, color='tab:green', ls=':', lw=1.6, label='GT onset')
    ax.set_ylabel(f'{name}\n[{state[name]["unit"] if name in state else "?"}]', fontsize=8)
    ax.legend(fontsize=7, loc='upper left')

axs = axes[-1]
axs.step(t, pred_stage, where='post', color='tab:blue', lw=1.4, label='predicted')
axs.step(t, gt_stage,   where='post', color='tab:green', lw=1.4, ls=':', label='ground truth')
axs.set_yticks(range(len(STAGE_NAMES))); axs.set_yticklabels(STAGE_NAMES, fontsize=7)
axs.set_ylabel('stage'); axs.set_xlabel('time [s]'); axs.legend(fontsize=8, loc='upper left')
fig.suptitle(f'scene2 (unseen) — predicted vs ground-truth lockbox stage   '
             f'(acc {acc*100:.1f}%, stage MAE {mae:.2f})', y=1.001)
plt.tight_layout(); plt.savefig(EVAL_OUT/'stage_eval.png', dpi=120, bbox_inches='tight'); plt.show()
print(f' {EVAL_OUT/"stage_eval.png"}')

 /home/kenny/HTCV/data/validation_eval/scene2/stage_eval.png
